# Hipotesis multivariada propuesta por catedra:
A igual capacidad de huéspedes, un alojamiento se alquila más caro si
garantiza mayor independencia: tener una cama para cada persona (no
compartir colchón), mayor cantidad de baños por huésped, y reserva
automática.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

listing_dataset = pd.read_csv("listings (1).csv.gz")
calendar_dataset = pd.read_csv("calendar.csv.gz")

: 

In [ ]:
# Confirmamos que 'instant_bookable' solo tiene filas nulas
listing_dataset['instant_bookable'].value_counts(dropna=False)

In [ ]:
#copiamos el conjunto para preservarlo
preprocessed_dataset = listing_dataset.copy()

#borramos columnas con todos sus valores nulos
preprocessed_dataset = preprocessed_dataset.drop('neighborhood_overview', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_since', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_response_time', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_response_rate', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_acceptance_rate', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_thumbnail_url', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_neighbourhood', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_total_listings_count', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_verifications', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('neighbourhood', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('neighbourhood_group_cleansed', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('calendar_updated', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('instant_bookable', axis=1)
preprocessed_dataset = preprocessed_dataset.drop('listing_url',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_url',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_profile_id',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_profile_url',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_name',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('scrape_id',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('picture_url',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('host_id',axis=1)
preprocessed_dataset = preprocessed_dataset.drop('license',axis=1)

preprocessed_dataset.info()

### Variables involucradas en hipotesis:

 + instant_bookable (De momento no lo voy a tener en cuenta por que es una columna nula).
 + beds
 + bathrooms
 + accommodates
 + price ó price_quote_price_per_night (la segunda no hace falta la limpieza de caracteres).
 + bathrooms_text (se usa para identificar los baños compartidos).


### Analisis

La fila de ``'instant_bookable'`` es una fila nula en su totalidad por lo tanto, no va a ser tomada en cuenta en el analisis de la hipotesis.

In [ ]:
# Miramos cuantas filas vacias tiene cada variable a analizar

cols = ['beds', 'bathrooms', 'accommodates', 'price_quote_price_per_night']
preprocessed_dataset[cols].isna().mean().sort_values()


In [ ]:
cols = ['beds', 'bathrooms', 'accommodates', 'price_quote_price_per_night']

n_antes = len(preprocessed_dataset)
dataset_without_null = preprocessed_dataset.dropna(subset=cols)
dataset_without_null = dataset_without_null[(dataset_without_null['price_quote_price_per_night'] > 0) & (dataset_without_null['accommodates'] > 0)]

print(f"Filas antes: {n_antes} | después: {len(dataset_without_null)} | perdidas: {1 - len(dataset_without_null)/n_antes:.1%}")

`price_quote_price_per_night` tiene un 6 % de nulos, `beds` un 8,7 % y `bathrooms` un 11,6 %. Dado el tamaño de la base, se decide **eliminar las filas con nulos** en estas variables (y las de precio o capacidad iguales a cero), lo que implica perder un 14,9 % de las filas (quedan 25.256 de 29.685).

Para evaluar si esta decisión sesga el análisis, se compararon las medianas de precio: 101.419 pesos en las filas conservadas contra 92.253 en las descartadas que tienen precio, una diferencia de unos 9.000 pesos (alrededor del 9 %). Es un cambio leve, por lo que se considera razonable continuar sin esas filas, y queda registrado como limitación que las filas descartadas tienden a tener precios algo menores.

In [ ]:
descartadas = preprocessed_dataset.drop(dataset_without_null.index)
print(dataset_without_null['price_quote_price_per_night'].median(), descartadas['price_quote_price_per_night'].median())

### *Creacion de variables y exloratoria de las mismas*

Para analizar parte de la hipotesis no hace falta crear nuevas variables:
+ Tenemos primero `'camas_para_cada_huesped'`, con una condicion ``'beds'`` >= ``'accommodates'``.
+ Segundo ``'bath_per_huesped'`` = ``'bathrooms'`` / ``'accommodates'``. 

El hecho de que `'camas_para_cada_huesped'` sea una variable dicotomica puede llegar a dar falsos negativos por que la hipotesis dice "no compartir colchon", si en el alojamiento se haya una cama dos plazas esta variable nos indicara false ya que la variable beds cuenta camas no colchones.

In [ ]:
# Creo la variable de bath_per_huesped que no tiene tanto analisis ni inferencia como la otra
dataset_without_null['bath_per_huesped'] = dataset_without_null['bathrooms'] / dataset_without_null['accommodates']
dataset_without_null['bath_per_huesped'].describe()

dataset_without_null.groupby('accommodates')['bath_per_huesped'].agg(['mean', 'count'])


In [ ]:
dataset_without_null['cama_para_cada_uno'] = dataset_without_null['beds'] >= dataset_without_null['accommodates']
dataset_without_null.groupby('accommodates')['cama_para_cada_uno'].agg(['mean', 'count'])

In [ ]:
dataset_without_null['cama_para_cada_uno'].value_counts()

Tenemos 20502 False (81%) y 4754 True (19%), por mas que el porcentaje de True's parezca pequeño si lo comparamos con el otro lado, la cantidad de muestras es suficiente como para generar una conclusion fiable.

Usaremos las "muestras" con accommodates de entre 2-6 ya que son los que concentran la mayoria de las filas y el porcentaje de True's para mayor capacidad nos da muy poca cantidad de filas que no modificarian a el analisis el que se tomen o no. Asi como tambien todo alquiler para una persona tampoco lo tendremos en cuenta por que es algo trivial ya que todas tienen un colchon para una persona.


In [ ]:
df_comp = dataset_without_null[dataset_without_null['accommodates'].between(2, 6)]
print(len(df_comp), len(df_comp) / len(dataset_without_null))

Si nos quedamos con las filas que mencionamos aun asi mantendriamos el 92,85% de lo que ya teniamos (23450 filas).
Analicemos si el hecho de que el alojamiento sea mas caro solo por el hecho de contar con camas para cada uno de los huespedes.

In [ ]:
df_comp.groupby(['accommodates', 'cama_para_cada_uno'])['price_quote_price_per_night'].median().unstack()

In [ ]:
#Grafico que explica mejor la tabla

plt.figure(figsize=(9, 5))
sns.barplot(data=df_comp, x='accommodates', y='price_quote_price_per_night',
            hue='cama_para_cada_uno', estimator='median', errorbar=None)
plt.title('Precio mediano por noche según cama para cada huésped')
plt.xlabel('Cantidad de huéspedes')
plt.ylabel('Precio mediano por noche')
plt.legend(title='Cama para cada uno', labels=['No', 'Sí'])
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_comp, x='accommodates', y='price_quote_price_per_night',
            hue='cama_para_cada_uno')
plt.yscale('log')
plt.title('Distribución del precio por noche según cama para cada huésped')
plt.xlabel('Cantidad de huéspedes')
plt.ylabel('Precio por noche (escala log)')
plt.legend(title='Cama para cada uno', labels=['No', 'Sí'])
plt.show()

# Tengo un error en la leyenda del graf. lo intente acomodar pero me da problemas el seaborn

Como vemos no es asi y estaria rompiendo la hipotesis si solo decimos que el precio sube por la cantida de camas, en el unico caso donde eso se cumple es para accommodates = 3 y tampoco es que aumente por demasia el precio.

Cambiando a la variable `'bathrooms'`, tiremos un scaterplot para ver como se distribuye con respecto a la cantidad de huespedes.

In [ ]:
plt.figure(figsize=(9, 5))
sns.scatterplot(data=dataset_without_null, x='bath_per_huesped', y='price_quote_price_per_night',
                hue='accommodates', palette='tab10', alpha=0.3)
plt.yscale('log')
plt.title('Precio por noche vs. baños por huésped')
plt.xlabel('Baños por huésped')
plt.ylabel('Precio por noche (escala log)')
plt.show()

In [ ]:
df_comp.groupby(['accommodates', 'bathrooms'])['price_quote_price_per_night'].agg(['median', 'count'])

Analizando la metadata de AirBnB, cuando la variable `bathrooms` tiene valores con decimal quiere notar que hay un medio baño, ¿que es un medio baño?. Que tiene lo esencial para que sea un baño, un inodoro y una bacha, no cuenta con ducha ni bañera.
Podemos pensar que para los valores altos de la variable estamos hablando de ser un lugar con mas "medio baños" que con baños completos y viendo que, pero lo que me esta mostrando esto es primero que nada que claramente hay una tendencia de a mayor cantidad de baños mayor 

La variable `'bath_per_huesped'` acota en rangos mas legibles en %, siendo 1 el hecho de que hay un baño para cada huesped, si esta por debajo de 1 es que hay menos y si esta por encima es que hay mas baños que huespedes.

In [ ]:
# bathrooms_text nos da informacion de que tipo de baños tiene el alojamiento
df_comp['bathrooms_text'].value_counts().head(15)

``bathroom_text`` nos podria estar diciendo que en un alojamiento hay baños compartidos y baños privados, no se llega a especificar si dentro del lugar se hayan ambos tipos de baños sino que la aplicacion le obliga a poner al host si un baño es o compartido o privado.
Para unificar y simplicar filas podemos decir que las filas donde no se especifican el tipo de baños los tomamos como privados.

Hace falta tambien sacar las filas donde hay exageradamente mas baños que capacidad de huespedes, puede ser un error de tipeo u otra cosa pero sin embargo son casos exepcionales, es decir, que hay muy pocos y solo estan ahi para tirar la media para arriba.

In [ ]:
df_comp['banio_compartido'] = df_comp['bathrooms_text'].str.contains('shared', case=False, na=False)
print(df_comp['banio_compartido'].mean())
pd.crosstab(df_comp['banio_compartido'], df_comp['bath_per_huesped'] > 1)

In [ ]:
# Cantidad de filas que quedan si eliminamos los baños compartidos y los casos donde
# hay muchos mas baños que huespedes
df_comp2 = df_comp[(~df_comp['banio_compartido']) & (df_comp['bath_per_huesped'] <= 1)].copy()
print(len(df_comp2), len(df_comp2) / len(df_comp))

In [ ]:
df_comp2['tramo_banios'] = pd.cut(df_comp2['bath_per_huesped'], bins=[0, 0.25, 0.5, 1])

df_comp2.groupby(['accommodates', 'tramo_banios'], observed=True)['price_quote_price_per_night'].agg(['median', 'count'])

Los tramos podemos categorizarlos como:
+ (0.0, 0.25]: Pocos baños
+ (0.25, 0.5]: Intermedio
+ (0.5, 1]: Muchos baños

Vemos como claramente aislando el resto de variables involucradas, el que haya mas baños hace que el precio del lugar sea mayor, tambien por algo logico el hecho de que haya mas baños tambien quiere decir que el establecimiento es mas grande por ende ya sin necesidad de tener que analizar probabilisticamente podemos deducir que si bien hay una tendencia a mayor precio cuanto mas baños tambien el precio este inducido por el tamaño del alojamiento que conlleva el tener mas baños justamente.

**Baños compartidos.** Se excluyen del análisis los alojamientos con baño compartido (604 filas, 2,6 % de las 23.450 filas con 2 a 6 huéspedes), por tratarse de un caso distinto al que plantea la hipótesis. La variable `banio_compartido` se usó solo como filtro y no entró al modelo, por lo que no se estudió si un baño compartido abarata el alquiler a igual capacidad; queda como posible extensión del análisis.


In [ ]:
# Grafico de barras

plt.figure(figsize=(9, 5))
sns.barplot(data=df_comp2, x='accommodates', y='price_quote_price_per_night',
            hue='tramo_banios', estimator='median', errorbar=None)
plt.title('Precio mediano por noche según baños por huésped')
plt.xlabel('Cantidad de huéspedes')
plt.ylabel('Precio mediano por noche')
plt.show()

In [ ]:
# Boxplots

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_comp2, x='accommodates', y='price_quote_price_per_night',
            hue='tramo_banios')
plt.yscale('log')
plt.title('Distribución del precio por noche según baños por huésped')
plt.xlabel('Cantidad de huéspedes')
plt.ylabel('Precio por noche (escala log)')
plt.show()

-------------------------------------------------------------------------------------------------------------------------------

**Aca comienzo a hacer la regresion lineal multiple**

**Planteo del modelo y de las hipótesis**

Modelo (alojamientos de 2 a 6 huéspedes, sin baños compartidos):

$$\log(\text{precio}_i) = \beta_0 + \sum_{k=3}^{6} \delta_k \, \mathbb{1}(\text{accommodates}_i = k) + \beta_{cama}\,\text{cama\_para\_cada\_uno}_i + \beta_{baños}\,\text{bath\_per\_huesped}_i + \varepsilon_i$$

La capacidad de huéspedes entra como variable categórica (referencia: 2 huéspedes), de modo que los coeficientes de camas y baños se interpretan **a igual capacidad**, como pide la hipótesis. Como la hipótesis plantea que más independencia implica un precio mayor, se contrastan alternativas unilaterales:

- Baños: $H_0: \beta_{baños} \leq 0$ contra $H_1: \beta_{baños} > 0$.
- Camas: $H_0: \beta_{cama} \leq 0$ contra $H_1: \beta_{cama} > 0$.

Nivel de significación $\alpha = 0{,}05$. El modelo se estima por mínimos cuadrados ordinarios; luego de revisar los supuestos sobre los residuos, la inferencia final (los tests de las hipótesis) se realiza con errores estándar robustos (HC3), como se detalla más adelante. La tercera componente de la hipótesis, la reserva automática (`instant_bookable`), no puede contrastarse porque la columna está vacía.

In [ ]:
import numpy as np
import statsmodels.formula.api as smf

df_comp2['log_precio'] = np.log(df_comp2['price_quote_price_per_night'])
df_comp2['cama_para_cada_uno'] = df_comp2['cama_para_cada_uno'].astype(int)

modelo = smf.ols(
    'log_precio ~ C(accommodates) + cama_para_cada_uno + bath_per_huesped',
    data=df_comp2
).fit()
print(modelo.summary())

import statsmodels.api as sm
sm.qqplot(modelo.resid, line='45')
plt.title('QQ-Plot de los residuos')
plt.show()

plt.scatter(modelo.fittedvalues, modelo.resid, alpha=0.3)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Valores ajustados')
plt.ylabel('Residuos')
plt.title('Valores ajustados vs. residuos')
plt.show()

In [ ]:
from scipy.stats import shapiro
from statsmodels.stats.diagnostic import het_breuschpagan

# Normalidad: Shapiro-Wilk sobre una muestra de 5.000 residuos
W, p_sw = shapiro(modelo_final.resid.sample(5000, random_state=42))
print(f"Shapiro-Wilk (muestra de 5.000 residuos): W = {W:.4f} | p = {p_sw:.2e}")

# Homocedasticidad: Breusch-Pagan
lm, p_bp, f_bp, p_f = het_breuschpagan(modelo_final.resid, modelo_final.model.exog)
print(f"Breusch-Pagan: LM = {lm:.1f} | p = {p_bp:.2e}")

Lo que suelta la tabla de OSL respalda lo analizado en la exploratoria de las variables y vemos como `'bath_per_huesped'` tiene un coeficiente de *1,29*. Por su parte, `'cama_para_cada_uno'` tiene un coeficiente de *-0.029* con un p-valor menor a 0.05 es decir, a igual capacida y baños por huesped, contar con una cama para cada huesped se asocia con un precio aproximadamente un 3% **menor**. El efecto es estadisticamente significativo pero chico y de signo contrario al planteado por la hipotesis, por lo que no la respalda en lo referido a las camas.

**Ajuste del modelo.** El $R^2$ es 0,175: la capacidad de huéspedes, las camas y los baños por huésped explican alrededor del 17,5 % de la variación del logaritmo del precio. Este valor mide cuánto del precio explica el modelo en conjunto, no si cada variable de la hipótesis tiene un efecto, por lo que no se usa para validarla ni refutarla: eso se evalúa con el signo y la significación de los coeficientes. Un $R^2$ bajo es esperable, porque el modelo deja afuera factores que influyen mucho en el precio (como el barrio, el tipo de alojamiento o la cantidad de dormitorios). Además, el QQ-plot muestra una cola derecha pesada: unos pocos precios extremadamente altos (posibles errores de carga) inflan la variación no explicada y bajan el $R^2$.

In [ ]:
df_comp2['resid'] = modelo.resid
extremos = df_comp2[df_comp2['resid'] > 4]
extremos[['accommodates', 'bathrooms', 'price_quote_price_per_night', 'resid']].sort_values('resid', ascending=False).head(15)
print(len(extremos))
df_comp2['price_quote_price_per_night'].quantile([0.5, 0.9, 0.99, 0.999, 1])

: 

In [ ]:
corte_995 = df_comp2['price_quote_price_per_night'].quantile(0.995)
print(f"Percentil 99,5: {corte_995:,.0f}")

for nombre, corte in [('10 millones', 1e7), ('p99,5', corte_995), ('p99', df_comp2['price_quote_price_per_night'].quantile(0.99))]:
    d = df_comp2[df_comp2['price_quote_price_per_night'] <= corte]
    m = smf.ols('log_precio ~ C(accommodates) + cama_para_cada_uno + bath_per_huesped', data=d).fit()
    print(f"{nombre:12} | filas {len(d)} | R2 {m.rsquared:.3f} | "
          f"cama {m.params['cama_para_cada_uno']:.3f} | baños {m.params['bath_per_huesped']:.3f} | "
          f"asimetría {m.resid.skew():.2f}")

Los precios por noche muestran un salto muy marcado en la cola superior: el percentil 99 es de 655.595 pesos, mientras que el percentil 99,9 llega a 76.019.870 (más de 100 veces mayor) y el máximo a 171.643.800. Valores de ese orden son muy poco plausibles para un alojamiento de 2 a 6 huéspedes y probablemente se deban a errores de carga del precio.

Para decidir dónde cortar se compararon tres criterios (excluir precios mayores a 10 millones, al percentil 99,5 y al percentil 99). Con el corte de 10 millones los residuos siguen muy asimétricos (asimetría de 1,32), mientras que el percentil 99 elimina más filas de las necesarias. Se eligió el **percentil 99,5**: se excluye el 0,5 % de alojamientos más caros (precio superior a 1.376.159 pesos por noche, 114 filas), quedando 22.650 filas. Se conservan los precios altos por debajo de ese umbral, entre los que puede haber alquileres de gama muy alta, algo que no se puede verificar con los datos.

Los resultados no dependen mucho de este criterio: el coeficiente de los baños por huésped varía entre 1,27 y 1,32 y el de las camas entre −0,022 y −0,025 según el corte, por lo que las conclusiones se mantienen. Queda registrado como limitación que este recorte se basa en el precio, que es la variable que se quiere explicar.

In [ ]:
d = df_comp2[df_comp2['price_quote_price_per_night'] <= corte_995].copy()
modelo_final = smf.ols('log_precio ~ C(accommodates) + cama_para_cada_uno + bath_per_huesped', data=d).fit()
print(modelo_final.summary())

sm.qqplot(modelo_final.resid, line='45')
plt.title('QQ-Plot de los residuos (modelo final)')
plt.show()

plt.scatter(modelo_final.fittedvalues, modelo_final.resid, alpha=0.3)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Valores ajustados')
plt.ylabel('Residuos')
plt.title('Valores ajustados vs. residuos (modelo final)')
plt.show()

: 

Tras excluir el 0,5 % de precios más altos (más de 1.376.159 por noche), el $R^2$ sube a 0,249 y el efecto de los baños por huésped casi no cambia (el coeficiente pasa de 1,29 a 1,31). El $R^2$ sigue siendo moderado, lo que indica que el modelo deja sin explicar buena parte de la variación del precio, pero eso no impide evaluar la hipótesis: a igual capacidad, más baños por huésped se asocia con un precio mayor, y el efecto es robusto al criterio de recorte (el coeficiente varía entre 1,27 y 1,32 según el corte). Las camas, en cambio, no respaldan la hipótesis, como se explicó antes.

PLos residuos del modelo final se alejan de la normalidad (cola derecha pesada, con una curtosis de 6,9), y la varianza del error podría no ser constante. Con 22.650 observaciones, la falta de normalidad casi no afecta a la inferencia, pero la heterocedasticidad sí puede distorsionar los errores estándar. Por eso se vuelve a estimar el modelo con errores estándar robustos (HC3) y se comparan con los convencionales.

In [ ]:
modelo_robusto = smf.ols('log_precio ~ C(accommodates) + cama_para_cada_uno + bath_per_huesped',
                         data=d).fit(cov_type='HC3')

comparacion = pd.DataFrame({
    'coef': modelo_final.params,
    'error_std_normal': modelo_final.bse,
    'error_std_robusto': modelo_robusto.bse,
    'p_normal': modelo_final.pvalues,
    'p_robusto': modelo_robusto.pvalues
}).round(4)

print(comparacion)

**Decisión**

| Componente | Coeficiente | Error estándar (HC3) | $t$ | $p$ unilateral | Decisión ($\alpha = 0{,}05$) |
|---|---|---|---|---|---|
| Baños por huésped | 1,3086 | 0,0402 | 32,6 | < 0,001 | Se rechaza $H_0$ |
| Cama para cada uno | −0,0235 | 0,0087 | −2,7 | 0,997 | No se rechaza $H_0$ |

Los baños por huésped tienen un efecto positivo y significativo sobre el precio, como plantea la hipótesis. En cambio, para las camas no se rechaza $H_0$: la estimación tiene signo contrario al esperado. Con un test bilateral, ese efecto negativo sería significativo ($p = 0{,}007$), pero no va en el sentido de la hipótesis.

**Magnitud del efecto de los baños**

El coeficiente de `bath_per_huesped` (1,31) está expresado en logaritmo del precio, por lo que cada aumento de $\Delta$ baños por huésped multiplica el precio esperado por $e^{1{,}31\,\Delta}$. Como la variable va de 0 a 1, conviene leerlo con saltos concretos, siempre a igual capacidad de huéspedes:

| Aumento de baños por huésped | Ejemplo | Efecto sobre el precio | IC 95 % |
|---|---|---|---|
| +0,10 | — | +14,0 % | 13,1 % a 14,9 % |
| +0,25 | alojamiento para 4: de 1 a 2 baños | +38,7 % | 36,0 % a 41,5 % |
| +0,50 | alojamiento para 2: de 1 a 2 baños | +92,4 % | 84,9 % a 100,1 % |

Es una asociación y no un efecto causal: tener más baños por huésped suele ir de la mano de un alojamiento más grande o de mayor categoría (más dormitorios, mejores terminaciones), que el modelo no controla. Por eso el efecto estimado probablemente incluye parte del efecto del tamaño y no solo el de la "independencia".

Para terminar, dado de que el conjunto es heterocedastico y no hay normalidad en los datos

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import kruskal, mannwhitneyu

d = d.copy()
d['log_precio'] = np.log(d['price_quote_price_per_night'])

# Quitar el efecto de la capacidad de huéspedes
m0 = smf.ols("log_precio ~ C(accommodates)", data=d).fit()
d['residuo'] = m0.resid

# Baños: Kruskal-Wallis entre tramos de baños por huésped
d['tramo_banos'] = pd.cut(d['bath_per_huesped'], [0, 0.25, 0.5, 1.0],
                          include_lowest=True, labels=['≤0,25', '0,25–0,5', '0,5–1'])
grupos = [g['residuo'].values for _, g in d.groupby('tramo_banos', observed=True)]
H, p = kruskal(*grupos)
print(f"Baños - Kruskal-Wallis: H = {H:.1f} | gl = {len(grupos) - 1} | p = {p:.2e}")
print(d.groupby('tramo_banos', observed=True)['residuo'].median().round(3))

# Camas: Mann-Whitney unilateral
mask = d['cama_para_cada_uno'].astype(bool)
U, p_u = mannwhitneyu(d.loc[mask, 'residuo'], d.loc[~mask, 'residuo'], alternative='greater')
print(f"Camas - Mann-Whitney (H1: con cama > sin cama): U = {U:.0f} | p = {p_u:.4f}")

Los coeficientes son los mismos que en el modelo anterior, porque los errores robustos modifican solo los errores estándar. Con HC3 estos aumentan levemente (el de los baños por huésped pasa de 0,029 a 0,040 y el de las camas de 0,0082 a 0,0087), pero **las conclusiones no cambian**: ambos efectos siguen siendo significativos ($p < 0{,}001$ para los baños y $p = 0{,}007$ para las camas). 

**Conclusión general**

La hipótesis de la cátedra plantea que, a igual capacidad de huéspedes, un alojamiento se alquila más caro si garantiza mayor independencia (una cama para cada huésped, más baños por huésped y reserva automática). Los resultados la respaldan **solo parcialmente**:

- **Baños por huésped: se respalda.** A igual capacidad, más baños por huésped se asocian con un precio mayor (coeficiente 1,31; IC 95 % robusto: 1,23 a 1,39; $p < 0{,}001$), con un efecto de unos +14 % por cada 0,1 baños por huésped adicionales.
- **Cama para cada huésped: no se respalda.** El coeficiente es −0,0235 (IC 95 % robusto: −0,041 a −0,006; $p = 0{,}007$ bilateral), equivalente a un precio 2,3 % menor. El efecto es significativo pero de signo contrario al planteado, por lo que no se rechaza $H_0$.
- **Reserva automática: no evaluable,** porque la columna `instant_bookable` está completamente vacía (29.685 de 29.685 valores nulos).

Por lo tanto, no se puede afirmar que la hipótesis se cumple en su totalidad: la evidencia apoya la relación entre baños por huésped y precio, pero no la de las camas, y la tercera componente queda sin contrastar. Dado que el modelo no controla el tamaño del alojamiento, estos resultados son asociaciones y no efectos causales.

**Limitaciones**

1. **Una de las tres componentes de la hipótesis no pudo evaluarse.** La reserva automática (`instant_bookable`) está completamente vacía en el dataset (29.685 de 29.685 valores nulos), por lo que solo se contrastaron las camas y los baños.
2. **La variable de camas es una aproximación.** `cama_para_cada_uno` vale 1 cuando `beds` ≥ `accommodates`, pero `beds` cuenta camas y no colchones: una cama doble figura como una sola cama aunque duerman dos personas. Esto puede clasificar como "sin independencia" a alojamientos que en la práctica sí la tienen, y debilitar el efecto estimado.
3. **Variables omitidas.** El modelo no controla el tamaño del alojamiento (dormitorios), el tipo de alojamiento ni el barrio, que influyen en el precio y se relacionan con los baños por huésped. Por eso el efecto de los baños puede incluir parte del efecto del tamaño o la categoría del alojamiento, y el $R^2$ es moderado (0,249). Los resultados son asociaciones y no efectos causales.
4. **Alcance acotado de la muestra.** El análisis usa 22.650 de las 29.685 filas originales (76,3 %): se excluyeron los nulos, los alojamientos de menos de 2 o más de 6 huéspedes, los baños compartidos y el 0,5 % de precios más altos. Las filas descartadas por nulos tienen una mediana de precio algo menor (92.253 contra 101.419 pesos), por lo que las conclusiones valen para el tipo de alojamiento que quedó en la muestra.
5. **Recorte sobre la variable de respuesta.** Excluir los precios más altos (más de 1.376.159 pesos por noche) puede sesgar levemente las estimaciones. Los coeficientes se mantienen estables entre los distintos cortes probados (baños entre 1,27 y 1,32; camas entre −0,022 y −0,025), lo que indica que las conclusiones no dependen de ese criterio.
6. **Supuestos del modelo.** Los residuos presentan cola derecha pesada (curtosis de 6,9) y la varianza del error parece no ser constante según el gráfico de residuos. Por eso se usaron errores estándar robustos (HC3), con los que las conclusiones se mantienen. La falta de normalidad tiene poco impacto con un tamaño de muestra tan grande.
7. **El precio es el publicado.** Se analiza el precio por noche informado en la publicación, que no necesariamente coincide con el efectivamente cobrado.